<a href="https://colab.research.google.com/github/valliansayoga/ey-data-challenge-2025/blob/master/EY2025_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
pd.options.display.max_columns = None

In [2]:
scl_mapping = {
    0: 'no_data',
    1: 'saturated_or_defective_pixel',
    2: 'topographic_casted_shadows',
    3: 'cloud_shadows',
    4: 'vegetation',
    5: 'not_vegetated',
    6: 'water',
    7: 'unclassified',
    8: 'cloud_medium_probability',
    9: 'cloud_high_probability',
    10: 'thin_cirrus',
    11: 'snow_or_ice'
}
scl_mapping

{0: 'no_data',
 1: 'saturated_or_defective_pixel',
 2: 'topographic_casted_shadows',
 3: 'cloud_shadows',
 4: 'vegetation',
 5: 'not_vegetated',
 6: 'water',
 7: 'unclassified',
 8: 'cloud_medium_probability',
 9: 'cloud_high_probability',
 10: 'thin_cirrus',
 11: 'snow_or_ice'}

In [3]:
pd.read_csv("Train_Final.csv").columns

Index(['Longitude', 'Latitude', 'datetime', 'UHI Index', 'is_a_building',
       'nearest_building_distance', '10m_nearby_building_count',
       '20m_nearby_building_count', '30m_nearby_building_count',
       '40m_nearby_building_count',
       ...
       'ndvi_median', 'trad_median', 'ndwi_std', 'ndmi_var', 'drad_var',
       'emsd_min', 'atran_mean', 'qa_radsat_min', 'qa_radsat_median',
       'lwir_mean'],
      dtype='object', length=179)

In [18]:
a = ExtraTreesRegressor()
a.fit([[1],[2],[3]], [[4],[5],[6]])

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


ExtraTreesRegressor()

In [21]:
a.feature_importances_.shape

(1,)

In [28]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    insample = r2_score(y_train, model.predict(X_train))
    outsample = r2_score(y_test, model.predict(X_test))
    return insample, outsample

def add_features(df):
    # Existing features
    stats = ["median", "mean", "min", "max", "var", "std"]
    epsilon = 1e-7

    count_cols = df.columns[df.columns.str.contains("count")]
    for col in count_cols:
        divider = int(col.split("_")[0].replace("m", "")[:-1])
        df[f"{col}_density_per_{divider}m"] = df[col] / divider

    for stat in stats:
        df[f"{stat}_evi_x_lwir"] = df[f"evi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_x_lwir"] = df[f"ndbi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_/_bldg_dnsty"] = df[f"ndbi_{stat}"] / df[f"building_density"].add(epsilon)
        df[f"{stat}_ndbi_/_ndwi"] = df[f"ndbi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_ndvi"] = df[f"ndbi_{stat}"] / df[f"ndvi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_evi"] = df[f"ndbi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_wvp_/_lwir"] = df[f"wvp_{stat}"] / df[f"lwir_{stat}"].add(epsilon)
        df[f"{stat}_infra_red_combo"] = df[f"nir08_{stat}"] * df[f"swir16_{stat}"] * df[f"swir22_{stat}"]
        df[f"{stat}_relative_ndvi"] = df[f"ndvi_{stat}"] / (df[f"ndvi_{stat}"].max() + epsilon)
        df[f"{stat}_relative_ndwi"] = df[f"ndwi_{stat}"] / (df[f"ndwi_{stat}"].max() + epsilon)
        df[f"{stat}_ndvi_ndwi_ratio"] = df[f"ndvi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndvi_evi_ratio"] = df[f"ndvi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_ndwi_evi_ratio"] = df[f"ndwi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_ndvi_ndwi_diff"] = df[f"ndvi_{stat}"] - df[f"ndwi_{stat}"]
        df[f"{stat}_ndvi_evi_diff"] = df[f"ndvi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_ndwi_evi_diff"] = df[f"ndwi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_combined_spectral_index"] = (df[f"ndvi_{stat}"] + df[f"ndwi_{stat}"] + df[f"evi_{stat}"]) / 3

    df["normalized_distance_range"] = df.distance_range / df.average_distance.add(epsilon)
    df["normalized_distance_variation"] = df.distance_variation / df.average_distance.add(epsilon)
    df["avg_nearby_building_count_per_10m"] = df["50m_nearby_building_count"] / 5
    df["distance_building_size_interaction"] = df.nearest_building_distance * df.nearest_building_size
    df["distance_building_density_interaction"] = df.nearest_building_distance * df.building_density
    df["average_distance_squared"] = df.average_distance ** 2
    df["nearest_building_distance_squared"] = df.nearest_building_distance ** 2
    df["log_building_area_density"] = np.log(df.building_area_density.add(epsilon))
    df["log_nearest_building_distance"] = np.log(df.nearest_building_distance.add(epsilon))
    df["distance_std_to_mean_ratio"] = df.std_distance / df.average_distance.add(epsilon)
    df["distance_variation_to_range_ratio"] = df.distance_variation / df.distance_range.add(epsilon)

    # Dangerous BOCOR feature!
    return df

def create_train(df_features, scaler, target="UHI Index", train_size=0.8, indices=None):
    print("Removing duplicates...")
    rows_before = df_features.shape[0]
    check_dupl = df_features.columns[1:]
    df_features = df_features.drop_duplicates(subset=check_dupl, keep='first')
    rows_after = df_features.shape[0]
    print(f"Removed {rows_before-rows_after} duplicate rows!")

    X = df_features.drop(target, axis=1)
    y = df_features[target]

    print("Scaling...")
    if indices is not None:
        X_train, X_test, y_train, y_test = X.iloc[indices[0]], X.iloc[indices[1]], y.iloc[indices[0]], y.iloc[indices[1]]
    else:
        X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=train_size)

    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    print("Done")
    return X_train, X_test, y_train, y_test, scaler

def round1(models, scaler):
    to_drop = ["Latitude", "Longitude", "datetime"]
    target = "UHI Index"

    # Round 1 to get pareto + 1 features
    separator = "-"*66
    spaces = " "*24
    equals = "="*32

    print(spaces, "Starting round 1", spaces)
    print(separator)
    df = pd.read_csv("Train_Final.csv").drop(to_drop, axis=1, errors="ignore")
    df.columns = df.columns.str.replace("_median_median", "_median", regex=False)
    df = df.pipe(add_features)

    X_train, X_test, y_train, y_test, scaler = create_train(
        df,
        scaler,
    )

    for model in tqdm(models):
        insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
        model["insample"] = insample
        model["outsample"] = outsample

    results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
    print(equals, "Model Scores", equals)
    print(results.head(1))
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    importance = pd.DataFrame(
        {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
    ).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance["cumulative_importance"] = importance.Importance.cumsum() / importance.Importance.sum()
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return importance

def round2(models, scaler, pareto_threshold, importance):
    to_drop = ["Latitude", "Longitude", "datetime"]
    target = "UHI Index"
    separator = "-"*66
    spaces = " "*24
    equals = "="*32

    # Round 2 to get pareto + 1
    pareto = importance[importance.cumulative_importance <= pareto_threshold]
    last_index = pareto.index[-1] + 1
    pareto = pd.concat([pareto, importance.iloc[last_index:last_index+1]])
    print(equals, "Pareto Features + 1", equals)
    print(pareto)
    print(separator)

    print(spaces, "Starting round 2", spaces)
    print(separator)
    df = pd.read_csv("Train_Final.csv").drop(to_drop, axis=1, errors="ignore")
    df.columns = df.columns.str.replace("_median_median", "_median", regex=False)
    df = df.pipe(add_features)
    use_cols = [target, *pareto.Features]
    df = df.loc[:, use_cols]
    X_train, X_test, y_train, y_test, scaler = create_train(
        df,
        scaler,
    )

    for model in tqdm(models):
        insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
        model["insample"] = insample
        model["outsample"] = outsample

    results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
    print(equals, "Model Scores", equals)
    print(results.head(1))
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    importance = pd.DataFrame(
        {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
    ).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance["cumulative_importance"] = importance.Importance.cumsum() / importance.Importance.sum()
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return best_model, X_train

# Modelling

In [74]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, StackingRegressor
from sklearn.preprocessing import StandardScaler

# StackingRegressor()
max_features = 0.6
models = [
    # {"model": RandomForestRegressor(random_state=0, n_jobs=-1, max_features=max_features)),
    # {"model": RandomForestRegressor(250, max_features=max_features, random_state=0, n_jobs=-1)),
    # {"model": RandomForestRegressor(150, max_features=max_features, random_state=0, n_jobs=-1)),
    # {"model": ExtraTreesRegressor(max_features=max_features, warm_start=True, n_estimators=200, random_state=0, n_jobs=-1)},
    # {"model": ExtraTreesRegressor(max_features=max_features, warm_start=True, n_estimators=250, random_state=0, n_jobs=-1)},
    # {"model": ExtraTreesRegressor(max_features=max_features, warm_start=True, n_estimators=100, random_state=0, n_jobs=-1)},
    {"model": ExtraTreesRegressor(
        max_features=max_features,
        n_estimators=300,
        random_state=0,
        n_jobs=-1,
        # criterion="friedman_mse",
    )},
]
scaler = StandardScaler()
importance = round1(models, scaler)

# Best score thrshld 0.75
# 0.974421

                         Starting round 1                         
------------------------------------------------------------------
Removing duplicates...
Removed 0 duplicate rows!
Scaling...
Done


100%|██████████| 1/1 [01:38<00:00, 98.28s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features=0.6, random_s...       1.0   0.974205
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features=0.6, n_estimators=300, n_jobs=-1,
                    random_state=0)
------------------------------------------------------------------
================================ Feature Importance ================================
                     Features  Importance  cumulative_importance
0                std_distance    0.063291               0.063291
1            average_distance    0.053780               0.117071
2    average_distance_squared    0.052086               0.169157
3          distance_variation    0.047358               0.216514
4              distance_range    0.042159        

In [75]:
best_model, X_train = round2(models, scaler, pareto_threshold=0.5, importance=importance)
# 0.5  0.975467

================================ Pareto Features + 1 ================================
                             Features  Importance  cumulative_importance
0                        std_distance    0.063291               0.063291
1                    average_distance    0.053780               0.117071
2            average_distance_squared    0.052086               0.169157
3                  distance_variation    0.047358               0.216514
4                      distance_range    0.042159               0.258674
5                           atran_min    0.036816               0.295489
6                         urad_median    0.018415               0.313904
7                        atran_median    0.017836               0.331740
8   distance_variation_to_range_ratio    0.013074               0.344815
9                   coast_aerosol_max    0.009814               0.354629
10                           ndmi_var    0.009720               0.364349
11                           urad_max 

100%|██████████| 1/1 [00:11<00:00, 11.61s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features=0.6, random_s...       1.0   0.975206
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features=0.6, n_estimators=300, n_jobs=-1,
                    random_state=0)
------------------------------------------------------------------
================================ Feature Importance ================================
                             Features  Importance  cumulative_importance
0            average_distance_squared    0.064852               0.064852
1                  distance_variation    0.064826               0.129679
2                      distance_range    0.063860               0.193539
3                           atran_min    0.062919               0.256458
4        

# Predicting Submission

In [63]:
def create_submission(filename: str, model, scaler):
    sub_df = pd.read_csv("Submission_Final.csv")
    sub_df.columns = sub_df.columns.str.replace(
        "_median_median", "_median", regex=False
    )
    final_df = sub_df[["Latitude", "Longitude"]].copy()
    print("Predicting", sub_df.shape[0], "rows...")

    ############################
    sub_df = add_features(sub_df)

    # # # # # Comment if not used!
    # sub_df.scl_median = sub_df.scl_median.map(scl_mapping)
    # scl_ohe = ohe.transform(sub_df.loc[:, ["scl_median"]])
    # scl_ohe = pd.DataFrame(scl_ohe, columns=ohe.get_feature_names_out(["scl_median"]))
    # sub_df = pd.concat([sub_df.drop("scl_median", axis=1), scl_ohe], axis=1)

    to_predict = pd.DataFrame(
        scaler.transform(sub_df.loc[:, X_train.columns]),
        columns=X_train.columns
    )

    print("Predicting...")
    final_df["UHI Index"] = model.predict(to_predict)
    final_df.to_csv(filename, index=False)
    print("Done!")
    return
create_submission("BestModel_OvrKllRadius50_Pareto05_MxFtrs02.csv", best_model.model, scaler)

Predicting 1040 rows...
Predicting...
Done!


---